# ND-AWS main analysis — convergence figures and the iteration/AR table

Reads the datasets the main run notebook produces and draws one panel per Hamiltonian
class, comparing the noise-directed and standard warm-started ansatzes on the QPU against
the noiseless matrix-product-state reference. It then rebuilds the iterations-at-convergence
and approximation-ratio table from the same data and prints it beside the published values.

The amplitude-damping sweep is analyzed separately, in
`01b_ndaws_amplitude_damping_analysis.ipynb`.

Which datasets enter which panel is stated explicitly in the configuration cell — there is
no implicit dataset discovery. Reading, exclusions and panel styling come from
`helpers/ndaws_paper_analysis.py`, which both analysis notebooks share, so run this notebook
from its own folder.

**Writes**: `fig3_<panel>.pdf` into `./temp/figures` inside the notebook folder. Nothing is
written into the result store.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import numpy as np
import plotly.io
from plotly.graph_objs import Scatter
from plotly.subplots import make_subplots

from helpers.ndaws_paper_analysis import (AR_FLOOR,
                                          FIGURE_DIR,
                                          LINE_WIDTH_BORDER,
                                          LINE_WIDTH_INSTANCES,
                                          MARKER_SIZE_CONVERGED,
                                          MARKER_SIZE_INSTANCES,
                                          finish_panel,
                                          load_dataset,
                                          pad_to_common_length)

## Panel specifications

The main run notebook builds every dataset name it writes from one `RUN_ID`. The cell below
applies the same rule in reverse, so pointing this notebook at a study means setting
`RUN_ID` to the value that study was written under — no dataset name is typed out by hand
and no dataset is discovered implicitly.

One study is three runs of the run notebook under one `RUN_ID`: the standard ansatz on the
QPU, the noise-directed ansatz on the QPU, and the noiseless matrix-product-state reference.
`FIG3_PANELS` ends up holding that triple per Hamiltonian class, plus the `backend` of the
two QPU members; the reference is always a simulation under `aer`.

`FIG3_CLASSES` carries the phase-separator fraction each class was run at, because the name
holds a `k=` token whenever that fraction is not 1.0.

Styling keys off the dataset names themselves: `mps` selects the grey reference line,
`no-gauges-version` the standard ansatz, `hdls` the darker shade.

In [ ]:
# Set this to the RUN_ID the study was written under.
RUN_ID = 'run-01'

BACKEND_NAME = 'ibm_boston'

ADD_HDLS = True
MPS_BOND_DIMENSION = 40
MPS_TIME_BLOCK_SIZE = 1.0

# Panel key -> the class token the run notebook writes into the name, and the
# phase-separator fraction the two QPU arms of that class were run at.
FIG3_CLASSES = {'ER-10': ('ER-0.1', 0.25),
                'ER-20': ('ER-0.2', 0.1),
                'RG-3': ('RG-3', 0.8)}


def build_dataset_name(class_token, arm, time_block_size, run_id=RUN_ID):
    """The naming rule of `00a_ndaws_main_run.ipynb`, read back.

    `arm` is 'standard', 'nd' or 'mps'. The reference arm is a noiseless matrix-product-state
    simulation, so it carries the simulation tokens where the other two carry `QPU`.
    """
    if arm == 'mps':
        name = f'NDAWS_simulation_noiseless_mps_chi={MPS_BOND_DIMENSION}_'
    else:
        name = 'NDAWS_QPU_'
    if time_block_size != 1.0:
        name += f'k={time_block_size}_'
    name += f'{class_token}_'
    if arm == 'standard':
        name += 'no-gauges-version_'
    if ADD_HDLS:
        name += 'HDLS_'
    return name + run_id


FIG3_PANELS = {
    panel_key: {'standard': build_dataset_name(class_token, 'standard', time_block_size),
                'nd': build_dataset_name(class_token, 'nd', time_block_size),
                'mps': build_dataset_name(class_token, 'mps', MPS_TIME_BLOCK_SIZE),
                'backend': BACKEND_NAME}
    for panel_key, (class_token, time_block_size) in FIG3_CLASSES.items()
}

# A member written under a different RUN_ID is overridden here, after the dict is built; a
# member that was never run is set to None to leave it out of its panel:
#   FIG3_PANELS['RG-3']['mps'] = build_dataset_name('RG-3', 'mps', 1.0, run_id='run-07')

for _panel_key, _spec in FIG3_PANELS.items():
    print(f"{_panel_key:<6} {_spec['nd']}")

os.makedirs(FIGURE_DIR, exist_ok=True)


## Reading a dataset

`load_dataset`, imported above from `helpers/ndaws_paper_analysis.py`, builds the analyzer
for one dataset, reads its overview results and returns the per-run convergence data
together with the analyzer's own printed output. The same module holds the experiment-set
ids excluded at read time, so both analysis notebooks drop exactly the same runs.

The returned `sets_per_instance` entry counts the experiment sets each Hamiltonian instance
contributed. With `include_only_best_run=True` the analyzer picks, per instance, the best
set over **all** sets of that instance, so an instance holding more than three sets is a
best-of-more.

## Plotting

The panels draw `1 - AR` on a reversed logarithmic axis, so better is up, with a floor at
`1e-3`. A run shorter than the longest in its panel is padded with its own best
approximation ratio, which is the convention the table below also uses.

Each panel draws one thin line per instance plus a thick mean. The axes, the fonts and the
padding rule come from the shared module; only the colours and the trace names are set
here.

In [ ]:
def fig3_trace_name(dataset_name):
    hdls = 'hdls' in dataset_name.lower()
    if 'mps' in dataset_name.lower():
        chi = dataset_name.split('mps_chi=')[1].split('_')[0]
        return f"MPS (χ={chi})+HDQS" if hdls else f"MPS (χ={chi})"
    base = 'Standard' if 'no-gauges-version' in dataset_name else 'Noise-Directed'
    return f"{base}+HDQS (QPU)" if hdls else f"{base} (QPU)"


def fig3_style(dataset_name):
    hdls = 'hdls' in dataset_name.lower()
    if 'mps' in dataset_name.lower():
        return dict(color='rgba(78, 78, 78)' if hdls else 'rgba(128, 128, 128)',
                    symbol='circle', is_reference=True)
    if 'no-gauges-version' in dataset_name:
        return dict(color='rgba(170, 0, 10)' if hdls else 'rgba(220, 20, 60)',
                    symbol='x', is_reference=False)
    return dict(color='rgba(15, 55, 175)' if hdls else 'rgba(65, 105, 225)',
                symbol='circle', is_reference=False)


def plot_fig3_panel(panel_key, subfig_letter, loaded_members):
    merged = make_subplots(rows=1, cols=1)
    max_x_global = np.max([np.max([len(ys) for ys in m['all_ys']]) for m in loaded_members])

    for member in loaded_members:
        dataset_name = member['dataset_name']
        style = fig3_style(dataset_name)
        color_main_1 = style['color'][:-1] + ', 1.0)'
        color_main = f'{color_main_1[:-4]}1.0)'
        name_plot = fig3_trace_name(dataset_name)

        all_ys_extended = 1. - pad_to_common_length(member['all_ys'])
        all_ys_extended = np.clip(all_ys_extended, AR_FLOOR, None)
        all_ys_plot = [[np.max([AR_FLOOR, 1. - y]) for y in ys] for ys in member['all_ys']]

        xs = list(range(0, all_ys_extended.shape[1]))
        mean_ys = np.mean(all_ys_extended, axis=0)

        merged.add_trace(Scatter(x=xs,
                                 y=mean_ys,
                                 mode='lines' if style['is_reference'] else 'lines+markers',
                                 name=f'{name_plot}-Mean',
                                 line_dash='dot' if style['is_reference'] else 'solid',
                                 line_width=LINE_WIDTH_BORDER,
                                 line_color=color_main_1,
                                 marker_color=color_main_1,
                                 marker_size=MARKER_SIZE_INSTANCES * 2,
                                 legendgroup=name_plot,
                                 showlegend=False))

        if style['is_reference']:
            continue

        marker_dict = dict(symbol=style['symbol'], color=color_main)
        line_dict = dict(dash='dot', color=color_main)

        for _idx, ys in enumerate(all_ys_plot):
            merged.add_trace(Scatter(x=list(range(0, len(ys))),
                                     y=ys,
                                     mode='lines',
                                     line=line_dict,
                                     line_width=LINE_WIDTH_INSTANCES,
                                     marker=marker_dict,
                                     marker_size=MARKER_SIZE_INSTANCES,
                                     name=name_plot,
                                     legendgroup=name_plot,
                                     showlegend=_idx == 0), row=1, col=1)

            _best_index = int(np.argmin(ys))
            merged.add_trace(Scatter(x=[_best_index],
                                     y=[ys[_best_index]],
                                     mode='markers',
                                     marker=marker_dict,
                                     marker_size=MARKER_SIZE_CONVERGED,
                                     marker_symbol='star',
                                     legendgroup=name_plot,
                                     showlegend=False), row=1, col=1)

    merged.update_layout(xaxis_title=r"Iteration", yaxis_title=r"1-AR (log scale)")
    merged.update_xaxes(range=(-0.5, max_x_global + 1))

    return finish_panel(merged,
                        panel_key=panel_key,
                        subfig_letter=subfig_letter,
                        yrange=np.log10(np.array([0.0007, 0.3][::-1])).tolist(),
                        xticks=np.linspace(0, 30, 7, endpoint=True).astype(int),
                        xrange=(0, 33),
                        height=int(1600 * 0.7),
                        width=int(2600 * 0.7))

## Read the QPU panels and draw them

In [ ]:
FIG3_DATA = {}

# The 100-qubit study reports, per instance, the best of its three runs. The damping sweep
# in 01b averages all three instead, so the convention is stated at the call rather than
# left to a default.
INCLUDE_ONLY_BEST_RUN = True

for _index, (panel_key, spec) in enumerate(FIG3_PANELS.items()):
    print('\n' + '=' * 30, panel_key, '=' * 30)
    members = []
    for role in ('standard', 'nd', 'mps'):
        dataset_name = spec.get(role)
        if dataset_name is None:
            continue
        is_reference = (role == 'mps')
        loaded = load_dataset(dataset_name=dataset_name,
                              backend_name='aer' if is_reference else spec['backend'],
                              simulation=is_reference,
                              include_only_best_run=INCLUDE_ONLY_BEST_RUN)
        loaded['role'] = role
        members.append(loaded)

    FIG3_DATA[panel_key] = members

    figure = plot_fig3_panel(panel_key=panel_key,
                             subfig_letter='abcdefg'[_index],
                             loaded_members=members)
    file_path = os.path.join(FIGURE_DIR, f'fig3_{panel_key}.pdf')
    plotly.io.write_image(figure, file_path, format='pdf', scale=1)
    print('wrote', file_path)

## Iterations at convergence and approximation ratio

The table is rebuilt here from the same `(all_xs_converged, all_ys)` the figures use, with
the library's own padding rule reproduced verbatim: a run shorter than the longest run of
its dataset is padded with **its own maximum** approximation ratio, up to
`max(all_xs_converged) + 1` entries. The +3-iteration extension applied to runs that ended
by sampling the optimum is already inside the returned data.

Each row is printed three ways: the value computed here, the published value, and the
difference. The library's own `ROW:` and `AR mean (+-SD) at iteration 10:` lines, captured
from the same call, are printed underneath — so a reconstruction error in this notebook is
separable from a mismatch between the stored data and the published table.

In [ ]:
PAPER_TABLE2 = {
    ('ER-10', 'standard'): (8, 20, 13.5, 14.1, 3.4, 0.961, 0.9998, 0.977, 0.981, 0.012, 0.968, 0.028),
    ('ER-10', 'nd'):       (10, 23, 14, 13.9, 3.8, 0.974, 1.0, 0.991, 0.990, 0.008, 0.983, 0.015),
    ('ER-10', 'mps'):      (6, 18, 9, 10.6, 4.0, 0.989, 1.0, 1.0, 0.996, 0.005, 0.993, 0.010),
    ('ER-20', 'standard'): (9, 25, 10.5, 13.3, 4.9, 0.974, 1.0, 1.0, 0.994, 0.009, 0.987, 0.016),
    ('ER-20', 'nd'):       (9, 18, 11, 12.1, 2.7, 0.969, 1.0, 0.997, 0.992, 0.010, 0.990, 0.009),
    ('ER-20', 'mps'):      (6, 12, 10, 9.7, 1.7, 0.976, 1.0, 0.998, 0.995, 0.007, 0.995, 0.007),
    ('RG-3', 'standard'):  (7, 13, 9.5, 9.9, 2.0, 0.982, 0.998, 0.991, 0.991, 0.005, 0.991, 0.005),
    ('RG-3', 'nd'):        (9, 17, 11.5, 11.5, 2.1, 0.989, 1.0, 0.996, 0.996, 0.004, 0.994, 0.005),
    ('RG-3', 'mps'):       (7, 15, 9.5, 9.8, 2.0, 0.987, 0.9998, 0.996, 0.995, 0.004, 0.994, 0.004),
}

ROLE_LABELS = {'standard': 'Standard IWS', 'nd': 'ND-AWS', 'mps': 'MPS'}

METRIC_NAMES = ['it min', 'it max', 'it med', 'it mean', 'it SD',
                'AR min', 'AR max', 'AR med', 'AR mean', 'AR SD',
                'AR@10 mean', 'AR@10 SD']


def table2_row(all_xs_converged, all_ys):
    max_iteration = np.max(all_xs_converged)

    all_ys_extended = []
    for ys in all_ys:
        ys2 = ys.copy()
        if len(ys2)<max_iteration+1:
            ys2 = ys2+[np.max(ys2)]*(max_iteration+1-len(ys2))
        all_ys_extended.append(ys2)

    all_ys_extended = np.array(all_ys_extended)

    final = all_ys_extended[:, -1]
    if all_ys_extended.shape[1] >= 11:
        ar10_mean = float(np.mean(all_ys_extended[:, 10]))
        ar10_std = float(np.std(all_ys_extended[:, 10]))
    else:
        ar10_mean, ar10_std = None, None

    return (float(np.min(all_xs_converged)), float(np.max(all_xs_converged)),
            float(np.median(all_xs_converged)), float(np.mean(all_xs_converged)),
            float(np.std(all_xs_converged)),
            float(np.min(final)), float(np.max(final)), float(np.median(final)),
            float(np.mean(final)), float(np.std(final)),
            ar10_mean, ar10_std)


def format_cell(value):
    return 'n/a' if value is None else f'{value:.4g}'


for panel_key, members in FIG3_DATA.items():
    for member in members:
        role = member['role']
        computed = table2_row(member['all_xs_converged'], member['all_ys'])
        paper = PAPER_TABLE2.get((panel_key, role))

        print('\n' + '-' * 70)
        print(f"{panel_key} / {ROLE_LABELS[role]}   ({member['dataset_name']}, "
              f"NoQ={member['number_of_qubits']}, runs={len(member['all_ys'])})")
        print(f"  {'metric':<12} {'computed':>12} {'paper':>12} {'difference':>12}")
        for name, value, paper_value in zip(METRIC_NAMES, computed,
                                            paper if paper is not None else [None] * len(METRIC_NAMES)):
            if value is None or paper_value is None:
                difference = 'n/a'
            else:
                difference = f'{value - paper_value:+.4g}'
            print(f"  {name:<12} {format_cell(value):>12} {format_cell(paper_value):>12} {difference:>12}")

        for line in member['analyzer_output'].splitlines():
            if line.startswith('ROW:') or line.startswith('AR mean (+-SD) at iteration 10:'):
                print('  library:', line)